In [ ]:
import os
import json
import sys


In [ ]:
os.environ["DECORD_EOF_RETRY_MAX"] = "20480"

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

In [ ]:
dir_path = "/scratch/yag1/ego4d_data/v2/clips"

mp4_files = [f for f in os.listdir(dir_path) if f.endswith(".mp4")]

print(f"Number of mp4 files: {len(mp4_files)}")

In [ ]:
SRC_PARENT_DIR = "/scratch/yag1/Qwen3-VL-Embedding" 
sys.path.append(SRC_PARENT_DIR)

In [ ]:
from src.models.qwen3_vl_embedding import Qwen3VLEmbedder

In [ ]:
MODEL_PATH = "/scratch/yag1/models/Qwen3-VL-Embedding-2B" # "your model path"

embedder = Qwen3VLEmbedder(
    model_name_or_path=MODEL_PATH,
    fps=30.0,
)

print("Embedding model loaded successfully!")

In [ ]:
import numpy as np
import cv2
from decord import VideoReader, cpu

In [ ]:
def read_video(filename: str) -> tuple[VideoReader, float]:
    """Reads a video file and returns a VideoReader object and its FPS."""
    # Open video (Lazy initialization; does not read frames into memory yet)

    vr = VideoReader(filename, ctx=cpu(0))
    fps = vr.get_avg_fps()

    return vr, fps


In [ ]:
def preprocess_video_decord(
    vr: VideoReader, fps: float, target_num_frames: int, target_frame_size: tuple[int, int], start_time: float, end_time: float
) -> np.ndarray:
    
    # 1. Calculate frame range bounds
    start_frame = min(int(start_time * fps), len(vr) - 2)
    end_frame = min(int(end_time * fps), len(vr) - 2)
    
    # 2. Linearly sample global frame indices directly
    frame_indices = np.linspace(
        start_frame, end_frame, num=target_num_frames, endpoint=False, dtype=np.int32
    )
    
    # 3. Decord fetches ONLY the targeted frames (massive speedup)
    frames = vr.get_batch(frame_indices).asnumpy() # Shape: (num_frames, H, W, C)
    
    num_frames, orig_h, orig_w, channels = frames.shape
    target_h, target_w = target_frame_size

    # Calculate the scaling factor required to fit the frame inside the target dimensions
    # Taking the min() automatically decides whether to scale by width or height
    scale = min(target_w / orig_w, target_h / orig_h)
    new_w = int(orig_w * scale)
    new_h = int(orig_h * scale)
    
    # Calculate the padding required to center the frame
    # (Using floor division // to get integer pixel coordinates)
    pad_top = (target_h - new_h) // 2
    pad_left = (target_w - new_w) // 2
    
    # Initialize a blank canvas array filled with zeros (black) of the target size
    resized_frames = np.zeros(
        (num_frames, target_h, target_w, channels), dtype=frames.dtype
    )
    
    for i in range(num_frames):
        # 5. Resize the frame while maintaining aspect ratio
        resized_frame = cv2.resize(frames[i], (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        
        # 6. Place the resized frame onto the center of the canvas
        resized_frames[i, pad_top:pad_top+new_h, pad_left:pad_left+new_w] = resized_frame
    
    return resized_frames

In [ ]:
TARGET_WIDTH = 1280
TARGET_HEIGHT = 720
TARGET_FRAME_SIZE = (TARGET_HEIGHT, TARGET_WIDTH)
TARGET_NUM_FRAMES = 16


In [ ]:
segments_by_duration = {}
for i in range(1, 11):
    with open(f"{i}_seconds_segments.json") as f:
        segments_by_duration[i] = json.load(f)

In [ ]:
from tqdm import tqdm
import torch
from PIL import Image


In [ ]:
for mp4_file in tqdm(mp4_files[12:]):
    mp4_name = os.path.splitext(mp4_file)[0]
    os.makedirs(f"{dir_path}/{mp4_name}/qwen_video_embeddings", exist_ok=True)

    mp4_path = os.path.join(dir_path, mp4_file)
    video_reader, video_fps = read_video(mp4_path)

    for i in range(1, 11):
        segments = segments_by_duration[i][f"{dir_path}/{mp4_name}.mp4"]

        for idx, segment in enumerate(segments):
            start_time = segment[0]
            end_time = segment[1]
            
            frames = preprocess_video_decord(video_reader, video_fps, TARGET_NUM_FRAMES, TARGET_FRAME_SIZE, start_time, end_time)
            
            seg_id = f"{i}_seconds_segment_{idx}"
            
            pil_frames = [Image.fromarray(frame) for frame in frames]
            
            embedding = embedder.process([{"video": pil_frames}])
            
            embedding = embedding.squeeze(0) 
            
            torch.save(embedding, f"{dir_path}/{mp4_name}/qwen_video_embeddings/{seg_id}.pt")
    
    print(f"Finished saving embeddings for {mp4_file}")